In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y infernal
import os, requests, pathlib, time, random
from google.colab import files


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,966 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,889 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.

In [ ]:
# Collect desired parameters from user. Comma separated target RFAM IDs, and the desired # of synth. seqs.
# Our workflow:
#RF04331, RF04135, RF04185
#1000
#10000

#TODO: Add parameterized support to eliminate need to supply params via input...

rf_ids = input("Enter target Rfam IDs separated by commas (e.g., RF01234, RF01235): ")
rf_ids = [x.strip().upper() for x in rf_ids.split(",")]
num_seq = int(input("Enter the desired number of synthethic sequences to generate for each family: "))
num_subseq = int(input("Enter the desired number of sub-sequences to sample from each synthetic family: "))

# define the data path
RAW_DIR = pathlib.Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)


Enter target Rfam IDs separated by commas (e.g., RF01234, RF01235): RF04331, RF04135, RF04185
Enter the desired number of synthethic sequences to generate for each family: 1000
Enter the desired number of sub-sequences to sample from each synthetic family: 10000


# Step 1: Synthetic Data Generation

**Infernal Sequence Generation Overview**

* OK, so now we have the 3 RFAM families selected from Ilia's investigation (Step 0)
* Now, we need to feed the .cm files from each of those families into our data pipeline
* User will provide those 3 RFAM IDs

**RFAM API**

* We will use RFAM's API to download the covariance model for each of the target families.

**cmemit**

* cmemit takes in the .cm file (covariance model) and will provide us with a specified number of synthetic sequences which match the provided family

* For now, we should say that 1,000 sequences are sufficient, but maybe we will need more.

* Input: .cm file for the desired family
* Output: .fasta file containing N sequences which match the provided family, where N can be configured by the user

In [ ]:
# For each family:
    # Download the family's covariance model if it does not exist
    # Call cm_emit on that covariance model, and save the resulting .fasta file

for rf_id in rf_ids:
    print(f"\n=========")
    print(f"Processing {rf_id}")
    print("=========")

    # Create dir for rf_id if it doesn't exist already
    rf_id_dir = RAW_DIR / rf_id
    rf_id_dir.mkdir(parents=True, exist_ok=True)
    rf_id_sec_dir = RAW_DIR / rf_id/"secondary_structures"
    rf_id_sec_dir.mkdir(parents=True, exist_ok=True)
    rf_id_sec_dir = RAW_DIR / rf_id/"tertiary_structures"
    rf_id_sec_dir.mkdir(parents=True, exist_ok=True)

    # Check if rf_id.cm file is already present before downloading from RFAM API
    cm_file = RAW_DIR / f"{rf_id}/{rf_id}.cm"

    if cm_file.exists():
        print(f"{rf_id}.cm already downloaded, skipping download step.")
    else:
        print(f"Downloading {rf_id}.cm via rfam's API")
        url = f"https://rfam.org/family/{rf_id}/cm"
        !wget -q -O {cm_file} {url}

    # Verify file exists and has data
    if os.path.exists(cm_file) and os.path.getsize(cm_file) > 100:
      print(f"Success: {cm_file} is ready.")
      synthetic_fasta = RAW_DIR / f"{rf_id}/{rf_id}_{num_seq}_seqs.fasta"

      print(f"--- Generating {num_seq} sequences via Infernal cmemit ---")

      # Generate num_seq synth seqs as an unaligned fasta file
      !cmemit -N {num_seq} {cm_file} > {synthetic_fasta}

      # Validate that the .fasta file was created successfully
      if os.path.exists(synthetic_fasta) and os.path.getsize(synthetic_fasta) > 0:
        print(f"COMPLETE: Created {synthetic_fasta} ({os.path.getsize(synthetic_fasta)} bytes)\n")

        # This code was originally used to download the fasta files in-line from
        # inside the loop. We now have a /cell/ dedicated to downloading the full output data below...
        #files.download(output_fasta)
      else:
        print("Error: The output file was created but it is empty. Check the command above.")
    else:
      print(f"Error: {rf_id}.cm file does not exist. Perhaps an exception occurred during the download.")


Processing RF04331
Success: data/raw/RF04331/RF04331.cm is ready.
--- Generating 1000 sequences via Infernal cmemit ---
COMPLETE: Created data/raw/RF04331/RF04331_1000_seqs.fasta (92808 bytes)


Processing RF04135
Success: data/raw/RF04135/RF04135.cm is ready.
--- Generating 1000 sequences via Infernal cmemit ---
COMPLETE: Created data/raw/RF04135/RF04135_1000_seqs.fasta (175325 bytes)


Processing RF04185
Success: data/raw/RF04185/RF04185.cm is ready.
--- Generating 1000 sequences via Infernal cmemit ---
COMPLETE: Created data/raw/RF04185/RF04185_1000_seqs.fasta (125745 bytes)



# Step 2: Data Sampling

**Sampling Routine**

* OK, we now have N synthetic sequences, all belonging to the same family
* We need to randomly sample sub-sequences from the synthetic data each of random length 25-50 nucleotides long

* First, we pick a random synthetic sequence
* Secondly, we pick a random length L (25-50)
* Thirdly, we pick a random starting point T for the sub-sequence. This is a random selection between 0 and (Sequence_length - SubSequence_length)
* Fouthly, we grab the sub-sequence of length L starting from point T in our randomly selected sequence, and store it to an array

* We repeat this process until we have R sub-sequences, where R can be configured by the user

* Input: .fasta file containing synthetic sequences
* Output: .fasta file containing R randomly selected sub-sequences sampled from the input file

In [ ]:
# Simple function that reads sequences from a FASTA file, and returns them in an array

def read_fasta(filepath):
    sequences = []
    current_sequence = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: # Skip empty lines
                continue
            if line.startswith('>'): # Header line
                if current_sequence: # Save previous sequence if it exists
                    sequences.append(''.join(current_sequence))
                    current_sequence = []
            else: # Sequence line
                current_sequence.append(line)
        if current_sequence: # Add the last sequence after the loop finishes
            sequences.append(''.join(current_sequence))
    return sequences

print("Defined function: read_fasta_simple")

Defined function: read_fasta_simple


In [ ]:
# Utility function to save a list of sequences to a FASTA file
def save_sequences_to_fasta(sequences, filepath):
    with open(filepath, 'w') as f:
        for i, seq in enumerate(sequences):
            f.write(f">Sequence_{i+1}\n")
            f.write(f"{seq}\n")
    print(f"Successfully saved {len(sequences)} sequences to {filepath}")

print("Defined function: save_sequences_to_fasta")

Defined function: save_sequences_to_fasta


In [ ]:
for rf_id in rf_ids:
    print(f"\n=========")
    print(f"Sampling subsequences from {rf_id}")
    print("=========")

    synthetic_fasta = RAW_DIR / f"{rf_id}/{rf_id}_{num_seq}_seqs.fasta"
    subsequences_fasta = RAW_DIR / f"{rf_id}/{rf_id}_{num_subseq}.subseqs.fasta"

    # verify that the fasta file for this family exists before we proceed
    if os.path.exists(synthetic_fasta) and os.path.getsize(synthetic_fasta) > 0:
        print(f"Succesfully located synthetic sequence file {synthetic_fasta} ({os.path.getsize(synthetic_fasta)} bytes)")
    else:
        print(f"Could not locate the synthetic sequence file {synthetic_fasta}. Please verify that the synthetic sequences were succesfully generated by cmemit")

    seqs = read_fasta(synthetic_fasta)
    subseqs = []

    # take num_subseq samples from the syntehtic fasta, and then slice them into subsequence
    for i in range(num_subseq):
        # Choose a random sequence to sample from
        random_seq = seqs[random.randint(0, len(seqs) - 1)]

        # Choose a random length for the subsequence between 75 and 100
        random_len = random.randint(25, 40)

        # In order to take the substring, we must select a valid starting index
        # between 0 and (Sequence_length - SubSequence_length)
        seq_length = len(random_seq)
        if seq_length < random_len: # Skip if sequence is too short to sample from
            continue
        index = random.randint(0, (seq_length - random_len))

        # Take the subsample, and add it to subseqs
        subseq = random_seq[index : index + random_len]
        subseqs.append(subseq)

    # Save the generated subsequences to a FASTA file
    save_sequences_to_fasta(subseqs, subsequences_fasta)


Sampling subsequences from RF04331
Succesfully located synthetic sequence file data/raw/RF04331/RF04331_1000_seqs.fasta (92808 bytes)
Successfully saved 10000 sequences to data/raw/RF04331/RF04331_10000.subseqs.fasta

Sampling subsequences from RF04135
Succesfully located synthetic sequence file data/raw/RF04135/RF04135_1000_seqs.fasta (175325 bytes)
Successfully saved 10000 sequences to data/raw/RF04135/RF04135_10000.subseqs.fasta

Sampling subsequences from RF04185
Succesfully located synthetic sequence file data/raw/RF04185/RF04185_1000_seqs.fasta (125745 bytes)
Successfully saved 10000 sequences to data/raw/RF04185/RF04185_10000.subseqs.fasta


# Step 3: Structure Prediction

**RhoFold**

* Rhofold can take any sequence, and predict it's secondary (2D) and tertiary (3D) structures.
* We must use Rhofold to take each of our R sub-sequences, and predict both it's secondary and tertiary structures!

* Notably, Rhofold expects only a single sequence as an input
* We will loop through each sequence in the fasta file, and use Rhofold on each of them to get their secondary/tertiary structures

**Storing the results**

* Our data "looks" like this:

* Subsequence1  SubSequence2  ... SubSequenceR
* SecStructure1 SecStructure2 ... SecStructureR
* TerStructure1 TerStructure2 ... TerStructureR

* We have R sequences, each with a corresponding secondary structure (.ct) and a tertiary structure (.pdb)


In [ ]:
# Nasir & Andy's code to use Rhofold for structure prediction will go in this cell

Now that the full data pipeline is completed, you can use the following cell to download the entire data directory to your local file system.

In [ ]:
!du -sh data
files.download("data")

2.2M	data


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>